In [69]:
import ifcopenshell
import ifcopenshell.util as qz
import ifcopenshell.util.element as qx
model = ifcopenshell.open("wet_chamber_new.ifc")

# model = ifcopenshell.open("AC20-FZK-Haus.ifc")


# declare wall objects IfcWall
walls = model.by_type('IfcWall')
print(walls)


# wall = model.by_type('IfcWall')[0]
# print(wall.id())


# def get_opening_list_from_wall(wall):
#     """
#     Get the list of openings associated with the given wall.
 
#     :param wall: The wall object.
#     :type wall: IfcWall
 
#     :return: List of IfcOpening objects associated with the wall.
#     :rtype: list
#     """
#     opening_list = wall.HasOpenings
#     return opening_list

[#416=IfcWall('37Q2prSLvCROggJav7qAPR',#42,'Basic Wall:Generic - 200mm:362010',$,'Basic Wall:Generic - 200mm',#389,#412,'362010',.NOTDEFINED.), #526=IfcWall('37Q2prSLvCROggJav7qAOv',#42,'Basic Wall:Generic - 200mm:362104',$,'Basic Wall:Generic - 200mm',#504,#522,'362104',.NOTDEFINED.), #734=IfcWall('37Q2prSLvCROggJav7qATi',#42,'Basic Wall:Generic - 200mm:362285',$,'Basic Wall:Generic - 200mm',#712,#730,'362285',.NOTDEFINED.), #844=IfcWall('37Q2prSLvCROggJav7qASn',#42,'Basic Wall:Generic - 200mm:362352',$,'Basic Wall:Generic - 200mm',#822,#840,'362352',.NOTDEFINED.)]


In [70]:
# number of walls. For this model, "walls" only has 4 elements, which makes it easy
no_walls = len(walls)
print('Number of walls is ', no_walls)

print('Properties are ', walls[0])

wall = walls[0]

print('Loop start--------')
for i in range(0,len(wall)):
    print('Type is ',type(wall[i]),'. Value is ', wall[i])


# this is just a way to get direct property sets from the Ifcwall. Likely need to look at inverse relationship formulas
# ifcopenshell.entity_instance.entity_instance

Number of walls is  4
Properties are  #416=IfcWall('37Q2prSLvCROggJav7qAPR',#42,'Basic Wall:Generic - 200mm:362010',$,'Basic Wall:Generic - 200mm',#389,#412,'362010',.NOTDEFINED.)
Loop start--------
Type is  <class 'str'> . Value is  37Q2prSLvCROggJav7qAPR
Type is  <class 'ifcopenshell.entity_instance.entity_instance'> . Value is  #42=IfcOwnerHistory(#39,#5,$,.NOCHANGE.,$,$,$,1683877139)
Type is  <class 'str'> . Value is  Basic Wall:Generic - 200mm:362010
Type is  <class 'NoneType'> . Value is  None
Type is  <class 'str'> . Value is  Basic Wall:Generic - 200mm
Type is  <class 'ifcopenshell.entity_instance.entity_instance'> . Value is  #389=IfcLocalPlacement(#153,#388)
Type is  <class 'ifcopenshell.entity_instance.entity_instance'> . Value is  #412=IfcProductDefinitionShape($,$,(#394,#410))
Type is  <class 'str'> . Value is  362010
Type is  <class 'str'> . Value is  NOTDEFINED


In [ ]:
# check which walls have openings
for i in range(len(walls)):
    wallOpen = walls[i].HasOpenings
    if (wallOpen):
        print('Wall ',i,' has ', len(wallOpen),' openings')

Wall  1  has  3  openings
Wall  2  has  1  openings


In [84]:
# example obtaining opening dimensions from a single wall
for rel in walls[1].HasOpenings:
    opening = rel.RelatedOpeningElement
    print("Name is ", opening.Name)
    
    # this get property sets for the specific opening
    psets = qx.get_psets(opening)

    # This determines the exact name of the "proprties" used. Sometimes it could be pset, or qto when finding window dimensions
    # print(psets.keys())

    # since the dimensions could be stored in two different properties, we declare variables that can compute both i.e. one will be legit, the other will be empty
    qto = psets.get("Qto_OpeningElementBaseQuantities",{})
    pset_common = psets.get('Pset_OpeningElementCommon',{})

    width = qto.get("Width") or pset_common.get("Width")
    height = qto.get("Height") or pset_common.get("Height")
    depth = qto.get("Depth") or pset_common.get("Depth")
    print("WxHxD = ",width,'x',height,'x',depth, '\n')

Name is  Basic Wall:Generic - 200mm:362104
WxHxD =  1350.0 x 1450.0 x 200.0 

Name is  Basic Wall:Generic - 200mm:362104
WxHxD =  950.0 x 550.000000000001 x 200.0 

Name is  Basic Wall:Generic - 200mm:362104
WxHxD =  950.0 x 550.000000000001 x 200.0 



For my next trick, I'm going to try to obtain the wall property materials

In [97]:
for wall in walls:
    material = qx.get_material(wall)

    # first, work out the number of layers, and the materials used
    if (material.is_a("IfcMaterialLayerSetUsage")):
        layer_set = material.ForLayerSet
        print('Number of Wall Layers is ',len(layer_set.MaterialLayers))

        for layer in layer_set.MaterialLayers:
            # obtain material name
            mat_name = layer.Material.Name
            thickness = layer.LayerThickness
            print('Material is ',mat_name,'\n Thickness is ',thickness,'\n')


            # obtain thermal properties of the wall layer
            # https://standards.buildingsmart.org/IFC/RELEASE/IFC2x3/TC1/HTML/ifcmaterialpropertyresource/lexical/ifcthermalmaterialproperties.htm
            thermal_pset = qx.get_psets(layer.Material).get("Pset_MaterialThermal",{})
            conductivity = thermal_pset.get("ThermalConductivity")
            specific_heat = thermal_pset.get("SpecificHeatCapacity")
            # print(specific_heat)

Number of Wall Layers is  1
Material is  Default Wall 
 Thickness is  200.0 

Number of Wall Layers is  1
Material is  Default Wall 
 Thickness is  200.0 

Number of Wall Layers is  1
Material is  Default Wall 
 Thickness is  200.0 

Number of Wall Layers is  1
Material is  Default Wall 
 Thickness is  200.0 



In [ ]:
for wall in walls:
    material = qx.get_material(wall)

Aim:
To extract information from Ifc models using IfcOpenShell for use in designing the Fire Safety strategy

Objectives:
- Obtain property materials from a room, and use it to develop a design fire
- Obtain compartment size dimensions and opening, to calculate flashover conditions during a compartment fire
- Automatically compute travel distances within a building

OR...

Aim:
To evaluate travel distances within a building from Ifc models using IfcOpenShell

Objectives:
- Construct rooms within a building using nodes and graphs
- Compute travel distance between nodes using shortest straight line path
- Find longest paths of travel (this looks hard)
- Validate travel distance using pathfinder

https://en.wikipedia.org/wiki/Price%27s_model
